# PDFs do arXiv -> Markdown

Pega os PDFs que a DAG `arxiv_dag` baixou em `data/pdfs/<data>/` e converte cada um pra Markdown com o
**Docling** (o mesmo do `converter.py`, um conversor de arquivos para Markdown que eu fiz, que resolve colunas duplas). Os `.md` vão pra `data/markdown/`.

In [2]:
# docling converte o PDF (pesado, baixa uns modelos na primeira execução)
!pip install docling

In [3]:
import glob
import os
import time
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

## Extração: achando os PDFs que a DAG baixou

In [4]:
# a DAG grava os PDFs em data/pdfs/<data>/<id>.pdf
pdfs = sorted(glob.glob("data/pdfs/*/*.pdf"))
print(len(pdfs), "PDFs encontrados")
print(pdfs[:3])

49 PDFs encontrados
['data/pdfs/2026-10-04/2610.04824v1.pdf', 'data/pdfs/2026-10-04/2610.04825v1.pdf', 'data/pdfs/2026-10-04/2610.04829v1.pdf']


## Transformação: PDF -> Markdown

Cria o conversor uma vez (carregar os modelos do Docling é a parte mais cara) e reaproveita
em todos os PDFs. A diferença pro `converter.py` é só essa: lá ele é recriado a cada arquivo.

In [5]:
# True = OCR em tudo. Os PDFs do arXiv já têm texto embutido, então o OCR só deixa mais lento.
# Se algum PDF sair com texto estragado, liga de novo.
USAR_OCR = False

# limite pra testar rápido. None = converte todos
LIMITE = None

PASTA_MD = Path("data/markdown")
PASTA_MD.mkdir(parents=True, exist_ok=True)

opcoes = PdfPipelineOptions()
opcoes.do_ocr = USAR_OCR
opcoes.do_table_structure = True  # tabelas viram tabela em Markdown

conversor = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opcoes)}
)

In [6]:
alvo = pdfs if LIMITE is None else pdfs[:LIMITE]
falhas = []
inicio = time.time()

for i, caminho in enumerate(alvo, 1):
    destino = PASTA_MD / (Path(caminho).stem + ".md")

    # já convertido numa execução anterior: pula
    if destino.exists():
        continue

    try:
        resultado = conversor.convert(caminho)
        destino.write_text(resultado.document.export_to_markdown(), encoding="utf-8")
    except Exception as e:
        # um PDF quebrado não pode derrubar o lote inteiro
        falhas.append((caminho, str(e)))

    print("[%d/%d] %s (%.0f s)" % (i, len(alvo), destino.name, time.time() - inicio))

print("\nconvertidos:", len(list(PASTA_MD.glob("*.md"))), "| falhas:", len(falhas))
for caminho, erro in falhas:
    print("  ", caminho, "->", erro)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[8/49] 2610.04862v1.md (617 s)
[9/49] 2610.04864v1.md (704 s)
[10/49] 2610.04867v1.md (762 s)
[11/49] 2610.04868v1.md (805 s)
[12/49] 2610.04875v2.md (908 s)
[13/49] 2610.04882v1.md (1067 s)
[14/49] 2610.04889v1.md (1564 s)
[15/49] 2610.04893v1.md (1671 s)
[16/49] 2610.04898v2.md (1736 s)
[17/49] 2610.04902v1.md (1838 s)
[18/49] 2610.04905v1.md (2020 s)
[19/49] 2610.04907v1.md (2066 s)
[20/49] 2610.04911v1.md (2256 s)
[21/49] 2610.04914v1.md (2328 s)
[22/49] 2610.04915v1.md (2502 s)
[23/49] 2610.04916v1.md (2641 s)
[24/49] 2610.06977v1.md (2764 s)
[25/49] 2610.17503v1.md (2956 s)
[26/49] 2610.17521v1.md (3004 s)
[27/49] 2610.17523v1.md (3030 s)
[28/49] 2610.17526v1.md (3166 s)
2026-10-14 16:39:57,400 MatchingPostProcessor WARNING  5 of 457 pdf cells matched neither a row nor a column band of the 19x4 grid and were dropped from the table
[29/49] 2610.17531v1.md (3413 s)
[30/49] 2610.17540v1.md (3583 s)
[31/49] 2610.17541v1.md (3669 s)
[32/49] 2610.17544v1.md (4026 s)
[33/49] 2610.17549v

## Conferindo o resultado

In [7]:
# olha o começo de um .md pra ver se a ordem de leitura saiu certa
mds = sorted(PASTA_MD.glob("*.md"))
print(mds[0].name)
print(mds[0].read_text(encoding="utf-8")[:1500])

2610.04824v1.md
## AGENT BEHAVIOR AS CODE: EFFICIENT AND ROBUST LLM AGENTS WITH PROGRAMMATIC SPECIFICATIONS

Peng Qi, Chunliang Lyu, Gang Li† , Fabian Chan, Cheng Chang, Ignacio Cases† , Will Lu Uniphore

## ABSTRACT

AI agents based on foundation models (FMs) have demonstrated strong capabilities to perform complex open-ended tasks. However, they face some common challenges in practice: (a) agent behavior can deviate drastically even for semantically similar tasks, leading to catastrophically propagated errors; (b) high cost and latency due to FM calls, repeated in full whenever a task recurs with different inputs; (c) FMs' limited context and instruction following capability confine how well agents manage the ever-growing execution context and follow complex plans. We introduce Agent Behavior as Code Agent (ABCAGENT), which uses a symbolic program (e.g., Python code with potential neural functions) to fully specify the agent's behavior at runtime, with a powerful FM agent editing tha